# 00a — Setup: Schema, Volume & Raw Data

**Jackson and Jackson HR Digital**

First task in the pipeline. Makes the project self-contained on a fresh workspace so no manual CLI pre-staging is required.

### What this notebook does
1. Creates the target schema (`CREATE SCHEMA IF NOT EXISTS`) — the catalog must already exist.
2. Creates the `raw_data` managed volume (`CREATE VOLUME IF NOT EXISTS`).
3. Copies the bundled CSVs (`data/structured/*.csv`) to the volume root and the resume PDFs (`data/unstructured/*.pdf`) to `raw_data/resumes/`.

The `data/` folder is synced into the workspace alongside the notebooks by the bundle, so this reads from the deployed source and writes into the UC volume.

In [ ]:
dbutils.widgets.text("catalog",     "bx4",       "UC Catalog")
dbutils.widgets.text("schema",      "hrd_2030",  "UC Schema")
dbutils.widgets.text("volume_name", "raw_data",  "UC Volume Name")

In [ ]:
import os

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = "/Workspace" + _nb_path.rsplit("/notebooks", 1)[0]
try:
    from dotenv import load_dotenv
    load_dotenv(f"{project_root}/.env")
except ImportError:
    pass

# Widget takes priority (job passes it); fall back to .env for interactive runs
catalog     = dbutils.widgets.get("catalog")     or os.getenv("TARGET_CATALOG", "bx4")
schema      = dbutils.widgets.get("schema")      or os.getenv("TARGET_SCHEMA", "hrd_2030")
volume_name = dbutils.widgets.get("volume_name") or "raw_data"

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
data_root   = f"{project_root}/data"

print(f"Catalog     : {catalog}")
print(f"Schema      : {schema}")
print(f"Volume      : {catalog}.{schema}.{volume_name}")
print(f"Volume path : {volume_path}")
print(f"Data source : {data_root}")

In [ ]:
# ---------------------------------------------------------------------------
# Create schema and volume (idempotent). The catalog must already exist.
# ---------------------------------------------------------------------------
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print(f"✅ Schema ready: {catalog}.{schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume_name}")
print(f"✅ Volume ready: {catalog}.{schema}.{volume_name}")

In [ ]:
# ---------------------------------------------------------------------------
# Copy bundled data files into the volume (idempotent overwrite).
#   data/structured/*.csv   -> /Volumes/.../raw_data/
#   data/unstructured/*.pdf -> /Volumes/.../raw_data/resumes/
# Both workspace files and volumes are POSIX-accessible, so use shutil.
# ---------------------------------------------------------------------------
import os, shutil

src_structured   = f"{data_root}/structured"
src_unstructured = f"{data_root}/unstructured"
resumes_dir      = f"{volume_path}/resumes"
os.makedirs(resumes_dir, exist_ok=True)

csv_count = 0
for fname in sorted(os.listdir(src_structured)):
    if fname.endswith(".csv"):
        shutil.copyfile(f"{src_structured}/{fname}", f"{volume_path}/{fname}")
        csv_count += 1
        print(f"  CSV  -> {fname}")

pdf_count = 0
for fname in sorted(os.listdir(src_unstructured)):
    if fname.endswith(".pdf"):
        shutil.copyfile(f"{src_unstructured}/{fname}", f"{resumes_dir}/{fname}")
        pdf_count += 1

print(f"\n✅ Uploaded {csv_count} CSV(s) to {volume_path}")
print(f"✅ Uploaded {pdf_count} resume PDF(s) to {resumes_dir}")

In [ ]:
# ---------------------------------------------------------------------------
# Verify the volume contents before the bronze load runs.
# ---------------------------------------------------------------------------
root_files    = [f.name for f in dbutils.fs.ls(volume_path)]
resume_files  = [f.name for f in dbutils.fs.ls(f"{volume_path}/resumes") if f.name.endswith(".pdf")]
print(f"Volume root  : {root_files}")
print(f"Resume PDFs  : {len(resume_files)}")
assert any(f == "candidates.csv" for f in root_files), "candidates.csv missing from volume"
assert any(f == "job_requirements.csv" for f in root_files), "job_requirements.csv missing from volume"
assert len(resume_files) >= 1, "no resume PDFs found in volume"
print("\n✅ Setup complete — schema, volume, and data are ready for 00_load_bronze.")